#Dataset Dados Abertos - Íntegras de Acórdãos do Diário da Justiça (Acórdãos e Decisões)

Pre-processing dos datasets por ano (único ou lista - 2021 ... 2026), parametrizado por tipo de decisão (monocrática ou acórdão) e por tipo de processo.

### Resumo da Etapa de Preparação do Dataset

Este documento apresenta um script desenvolvido na linguagem Python (versão 3.12), cuja finalidade é concluir a elaboração do *dataset* central desta pesquisa de mestrado. A principal tarefa aqui executada consiste na consolidação e integração de um vasto volume de dados jurídicos (metadados e teores de decisões), os quais foram extraídos na fase anterior do projeto por meio de técnicas de *web scraping*. Através do cruzamento e unificação de bases distintas (sub-datasets), o código junta os dados brutos em um único arquivo, estruturando-os em um formato relacional consistente para formar o *dataset* principal da pesquisa. Para viabilizar esse processo de integração de dados, fez-se o uso intensivo da biblioteca Pandas (versão 2.2.2) na manipulação dos *dataframes*, com o apoio complementar do NumPy (versão 2.0.2) e PyTorch (versão 2.10.0).

##Importação de bibliotecas e declaração de variáveis globais

In [ ]:
NROWS = None # Para fins de testes, às vezes é necessário obter uma fração do dados. Coloque None para leitura completa.

TIPO_CONSTRUCAO = 'geral' #'acordaos', 'decisoes'
LISTA_TIPO_ARQUIVO = ['decisoes', 'acordaos']
ANO_CARGA = None
#'AREsp' #'REsp' #HC
CLASSE = 'HC'

#Não sendo informado o ANO
ANO_INI = 2021
ANO_FIM = 2025

import torch

# Check if a GPU (CUDA) is available
if torch.cuda.is_available():
    print("GPU is available!")
    print("Device Name:", torch.cuda.get_device_name(0))
else:
    print("GPU not available, using CPU.")

# This is the line you'd use in your script to send your model or data to the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

GPU not available, using CPU.
Using device: cpu


##Parametrização conforme ambiente de execução.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Definição do caminho base no Google Drive (ambiente COLAB)
base_path = '/content/drive/MyDrive/_Mestrado/Datasets/'

%run "/content/drive/MyDrive/_Mestrado/libs/preprocessing.ipynb"

Mounted at /content/drive
version 1.08.0


In [ ]:
from datetime import datetime, date
import pandas as pd

def cast_string_to_date(date_string: str) -> date:
    """
    Casts a string in 'DD/MM/YYYY' format to a datetime.date object.
    Handles nulls, empty strings, and whitespace robustly.
    """
    # 1. Check if it's null, not a string, or a NaN object
    if pd.isna(date_string) or not isinstance(date_string, str):
        return None

    # 2. Clean whitespace and check if empty
    clean_date = date_string.strip()
    if not clean_date:
        return None

    try:
        # 3. Standard conversion for dd/mm/yyyy
        return datetime.strptime(clean_date[0:10], '%d/%m/%Y').date()
    except (ValueError, TypeError):
        return None

##Carregar classe/rótulo para a decisão (Favorável ou Não ao Réu)

In [ ]:
# Define file paths
file_path = base_path + 'descricao-decisao-julgamento.csv'

# Load each CSV file into a dataframe
try:
    dfTipoDecisao = pd.read_csv(file_path, sep=';')
    print(f"Successfully loaded {file_path} into dataframe 'dfTipoDecisao'")
    display(dfTipoDecisao.head())
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")
except Exception as e:
    print(f"An error occurred while loading {file_path}: {e}")

Successfully loaded /content/drive/MyDrive/_Mestrado/Datasets/descricao-decisao-julgamento.csv into dataframe 'dfTipoDecisao'


,COD_TEOR_DECISAO,DESC_TEOR_DECISAO,DESC_RSULT_JGTO,RESULTADO_RESUMIDO,DECISAO_FAVORAVEL
0,1,Concedendo,"AGRAVO CONHECIDO EM PARTE E, NESSA PARTE, PROVIDO",PARCIALMENTE CONHECIDO E PROVIDO,S
1,1,Concedendo,Agravo regimental provido,PROVIDO,S
2,1,Concedendo,CONCEDENDO HABEAS CORPUS,CONCEDIDO,S
3,1,Concedendo,CONCEDENDO HABEAS DATA,CONCEDIDO,S
4,1,Concedendo,CONCEDENDO MANDADO DE SEGURANÇA,CONCEDIDA,S


##Carregar metadados dos acórdão ou decisões (Documento, Processo, Nro de Registro, Ministro, Teor).

In [ ]:
from datetime import datetime, date
import numpy as np
import pandas as pd

def load_metadados(ano: int, tipo_arquivo: str = 'acordaos', classe: str = ''):

  def cast_string_to_date(date_string: str) -> date:
      if pd.isna(date_string) or not isinstance(date_string, str):
          return pd.NaT

      clean_date = date_string.strip()
      if not clean_date:
          return pd.NaT

      try:
          return datetime.strptime(clean_date[0:10], '%d/%m/%Y').date()
      except (ValueError, TypeError):
          return pd.NaT

  def remove_blank(field):
    return field.strip() if isinstance(field, str) else field

  file_path = base_path + f'{ano}/metadados-{tipo_arquivo}-dj{ano}.csv'

  try:
      df_metadados = pd.read_csv(file_path, sep=';', nrows=NROWS)
      print(f"Successfully loaded {file_path} with {df_metadados.shape[0]} rows.")

      # Clean and filter by class
      df_metadados['PROCESSO'] = df_metadados['PROCESSO'].apply(remove_blank)
      df_metadados = df_metadados[df_metadados['PROCESSO'].astype(str).str.startswith(classe)].copy()

      if df_metadados.empty:
          print(f"No records found for class {classe}")
          return df_metadados

      # Robust date conversion
      df_metadados['DATAPUBLICACAO']   = pd.to_datetime(df_metadados['DATAPUBLICACAO'].apply(cast_string_to_date))
      df_metadados['DATARECEBIMENTO']  = pd.to_datetime(df_metadados['DATARECEBIMENTO'].apply(cast_string_to_date))
      df_metadados['DATADISTRIBUICAO'] = pd.to_datetime(df_metadados['DATADISTRIBUICAO'].apply(cast_string_to_date))

      # Days duration calculation
      df_metadados['NUM_DIAS_CONCLUSAO_PROCESSO'] = (df_metadados['DATAPUBLICACAO'] - df_metadados['DATARECEBIMENTO']).dt.days

      # Extract process info
      df_metadados['TIPOPROCESSO'] = df_metadados['PROCESSO'].apply(lambda x: str(x).split(" ")[0])
      df_metadados['NUMEROPROCESSO'] = df_metadados['PROCESSO'].apply(lambda x: str(x).split(" ")[1] if " " in str(x) else "")

      df_metadados.drop(columns=['PROCESSO'], inplace=True)

      # Standardize Minister column
      if 'NM_MINISTRO' in df_metadados.columns:
          df_metadados.rename(columns={'NM_MINISTRO': 'MINISTRO'}, inplace=True)

      # Remove columns explicitly requested to be ignored
      cols_to_drop = ['DESCRICAOMONOCRATICA']
      for col in cols_to_drop:
          if col in df_metadados.columns:
              df_metadados.drop(columns=[col], inplace=True)

      # Final text cleaning
      text_cols = ['TIPODOCUMENTO', 'NUMEROREGISTRO', 'RECURSO', 'TEOR', 'ASSUNTOS', 'MINISTRO']
      for col in text_cols:
          if col in df_metadados.columns:
              df_metadados[col] = df_metadados[col].apply(remove_blank)

      return df_metadados

  except Exception as e:
      print(f"An error occurred while loading {file_path}: {e}")
      return pd.DataFrame()

In [ ]:
#Teste isoaldo para observar resultados
# ano = 2021
# df2 = load_metadados(ano, 'acordaos', 'HC')
# df2.head()
# dfMetaDadosAcordao.head()

#dfDetalhesAcordao = load_dados_acordaos(ano)
#dfDetalhesAcordao.head()

# df2.head(3)

In [ ]:
# clear_object(df)
# ano = 2021
# tipo_arquivo = 'decisoes'
# classe = 'HC'
# estrutura_arquivo = 'metadados'
# print(f'Processing {ano} year...')
# print(f'   >> Processing MetaData for {ano}, {tipo_arquivo}, {CLASSE}...')

# file_path = base_path + f'{ano}/{estrutura_arquivo}-decisoes-dj{ano}.csv'
# print(file_path)

# df = pd.read_csv(file_path, sep=';', nrows=5)

# print(f"Successfully loaded {file_path} into dataframe 'dfMetaDadosDecisao' with {df.shape[0]} raws.")

# df.head()


In [ ]:
#Teste chamada direta
# LISTA_TIPO_ARQUIVO_TESTE = ['acordaos', 'decisoes']
# LISTA_ESTRUTURA_ARQUIVO = ['dados', 'metadados']
# LISTA_ANO_TESTE = [2021, 2022, 2023, 2024, 2025]

# for ano in LISTA_ANO_TESTE:
#   for tipo_arquivo in LISTA_TIPO_ARQUIVO_TESTE:

#     for estrutura_arquivo in LISTA_ESTRUTURA_ARQUIVO:
#       print(f'   >> Processing MetaData for {ano}, {tipo_arquivo}, {estrutura_arquivo} e {CLASSE}...')

#       file_path = base_path + f'{ano}/{estrutura_arquivo}-{tipo_arquivo}-dj{ano}.csv'
#       print(file_path)

#       df = pd.read_csv(file_path, sep=';', nrows=5)
#       print(f"Successfully loaded {file_path} into dataframe 'dfMetaDadosDecisao' with {df.shape[0]} raws.")
#       df.head()

In [ ]:
#Teste chamada direta dados textuais
# ano = 2021
# df_dados_teste = load_dados(ano, 'decisoes')
# df_dados_teste.head()

##Carregar detalhes das decisões

In [ ]:
def load_dados(ano: int, tipo_arquivo: str = 'acordaos'):
  # Define file paths
  file_path = base_path + f'{ano}/dados-{tipo_arquivo}-dj{ano}.csv'
  print(file_path)
  try:
      print("Loading dataset (optimized for testing)...")
      # Using nrows=50 to load only the necessary rows from disk
      df_dados_acordaos = pd.read_csv(file_path, sep=';', nrows=NROWS)
      #print(f"Successfully loaded the first 50 rows of {file_path}")

      # Drop rows where 'RELATORIO' is null in-place for better performance
      df_dados_acordaos.dropna(subset=['RELATORIO'], inplace=True)

      return df_dados_acordaos
  except FileNotFoundError:
      print(f"Error: File not found at {file_path}")
  except Exception as e:
      print(f"An error occurred while loading {file_path}: {e}")

In [ ]:
# df = load_dados(2021, TIPO_ARQUIVO)
# df

In [ ]:
def join_datasets(dfMetaDados, dfDetalhes, tipo_arquivo):

  def get_relatorio(seq_documento):
      row = dfDetalhes[dfDetalhes['SEQDOCUMENTO'] == seq_documento].head(1)
      return row['RELATORIO'].iloc[0] if not row.empty else None

  def get_tipo_decisao(teor_decisao):
      tipo_decisao_row = dfTipoDecisao[dfTipoDecisao['DESC_TEOR_DECISAO'] == teor_decisao].head(1)
      return tipo_decisao_row['DECISAO_FAVORAVEL'].iloc[0] if not tipo_decisao_row.empty else None

  # Mantendo apenas metadados básicos, decisão favorável e o relatório
  dfMetaDados['DECISAO_FAVORAVEL']  = dfMetaDados['TEOR'].apply(get_tipo_decisao)
  dfMetaDados['RELATORIO']  = dfMetaDados['SEQDOCUMENTO'].apply(get_relatorio)

  # Remove linhas sem relatório para manter a integridade do dataset
  dfMetaDados.dropna(subset=['RELATORIO'], inplace=True)

  return dfMetaDados

In [ ]:
import gc

def clear_object(obj):
  # Delete the individual dataframes to free up memory
  del obj

  # Explicitly call the garbage collector
  gc.collect()

  print("Memory for df_metadados and df_dados has been released.")

In [ ]:
# df_metadados.head()

In [ ]:
all_df_joins = []

inicio, fim = 0, 0
if ANO_CARGA:
  inicio, fim = ANO_CARGA, ANO_CARGA+1 # apenas um ano
else:
  inicio, fim = ANO_INI, ANO_FIM+1 # todo período

# Determina quais tipos processar com base em TIPO_CONSTRUCAO
if TIPO_CONSTRUCAO == 'geral':
    tipos_para_processar = LISTA_TIPO_ARQUIVO
else:
    tipos_para_processar = [TIPO_CONSTRUCAO]

print(f"Realizando carga de {inicio} a {fim-1} para o tipo: {TIPO_CONSTRUCAO}...")

for ano in range(inicio, fim):
    print(f'Processing {ano} year...')

    for tipo_arquivo in tipos_para_processar:
        print(f'   >> Processing MetaData for {ano}, {tipo_arquivo}, {CLASSE}...')
        df_metadados = load_metadados(ano, tipo_arquivo, CLASSE)

        if not df_metadados.empty:
            print('   >> Processing Text Data...')
            df_dados = load_dados(ano, tipo_arquivo)

            if df_dados is not None and not df_dados.empty:
                print('   >> Join metadata and text data (Relatório only)...')
                dfJoin = join_datasets(df_metadados, df_dados, tipo_arquivo)
                all_df_joins.append(dfJoin)

                clear_object(df_dados)
                clear_object(dfJoin)

        clear_object(df_metadados)

if all_df_joins:
    dataset = pd.concat(all_df_joins, ignore_index=True)
    clear_object(all_df_joins)

    # Garantir que DATAPUBLICACAO é datetime para ordenação correta
    dataset['DATAPUBLICACAO'] = pd.to_datetime(dataset['DATAPUBLICACAO'], errors='coerce')

    # Ordenar por DATAPUBLICACAO descendente
    dataset = dataset.sort_values(by='DATAPUBLICACAO', ascending=False, na_position='last')

    # Drop duplicates baseada no SEQDOCUMENTO
    dataset.drop_duplicates(subset=['SEQDOCUMENTO'], keep='first', inplace=True)

    # Reset index após ordenação e limpeza
    dataset.reset_index(drop=True, inplace=True)

    display(dataset.head())
else:
    print("Nenhum dado encontrado para os parâmetros informados.")

Realizando carga de 2021 a 2025 para o tipo: geral...
Processing 2021 year...
   >> Processing MetaData for 2021, decisoes, HC...
Successfully loaded /content/drive/MyDrive/_Mestrado/Datasets/2021/metadados-decisoes-dj2021.csv with 81256 rows.
   >> Processing Text Data...
/content/drive/MyDrive/_Mestrado/Datasets/2021/dados-decisoes-dj2021.csv
Loading dataset (optimized for testing)...
   >> Join metadata and text data (Relatório only)...
Memory for df_metadados and df_dados has been released.
Memory for df_metadados and df_dados has been released.
Memory for df_metadados and df_dados has been released.
   >> Processing MetaData for 2021, acordaos, HC...
Successfully loaded /content/drive/MyDrive/_Mestrado/Datasets/2021/metadados-acordaos-dj2021.csv with 12431 rows.
   >> Processing Text Data...
/content/drive/MyDrive/_Mestrado/Datasets/2021/dados-acordaos-dj2021.csv
Loading dataset (optimized for testing)...
   >> Join metadata and text data (Relatório only)...
Memory for df_metadado

,SEQDOCUMENTO,DATAPUBLICACAO,TIPODOCUMENTO,NUMEROREGISTRO,DATARECEBIMENTO,DATADISTRIBUICAO,MINISTRO,RECURSO,TEOR,ASSUNTOS,NUM_DIAS_CONCLUSAO_PROCESSO,TIPOPROCESSO,NUMEROPROCESSO,DECISAO_FAVORAVEL,RELATORIO
0,353105536,2025-12-31,DECISÃO,202505102484,2025-12-20,2025-12-23,HERMAN BENJAMIN,NaN,Negando,4355,11,HC,1063382,N,Cuida-se de Habeas Corpus com pedido de limina...
1,353128400,2025-12-31,DECISÃO,202505106071,2025-12-22,2025-12-26,HERMAN BENJAMIN,NaN,Negando,5897,9,HC,1063851,N,Trata-se de Habeas Corpus com pedido de limina...
2,353128835,2025-12-31,DECISÃO,202505109521,2025-12-23,2025-12-26,HERMAN BENJAMIN,NaN,Negando,3608,8,HC,1064262,N,Cuida-se de Habeas Corpus com pedido de limina...
3,353107646,2025-12-31,DECISÃO,202505102649,2025-12-21,2025-12-24,HERMAN BENJAMIN,NaN,Negando,4355,10,HC,1063449,N,Cuida-se de Habeas Corpus com pedido de limina...
4,352928478,2025-12-31,DECISÃO,202505059266,2025-12-18,2025-12-19,RIBEIRO DANTAS,NaN,Negando,10635,13,HC,1062801,N,"Trata-se de habeas corpus, com pedido de limin..."


In [ ]:
df_metadados.shape

(23546, 15)

In [ ]:
df_metadados.info()

<class 'pandas.core.frame.DataFrame'>
Index: 23546 entries, 0 to 24512
Data columns (total 15 columns):
 #   Column                       Non-Null Count  Dtype         
---  ------                       --------------  -----         
 0   SEQDOCUMENTO                 23546 non-null  int64         
 1   DATAPUBLICACAO               23546 non-null  datetime64[ns]
 2   TIPODOCUMENTO                23546 non-null  object        
 3   NUMEROREGISTRO               23546 non-null  int64         
 4   DATARECEBIMENTO              23546 non-null  datetime64[ns]
 5   DATADISTRIBUICAO             23546 non-null  datetime64[ns]
 6   MINISTRO                     23546 non-null  object        
 7   RECURSO                      22309 non-null  object        
 8   TEOR                         23546 non-null  object        
 9   ASSUNTOS                     23546 non-null  object        
 10  NUM_DIAS_CONCLUSAO_PROCESSO  23546 non-null  int64         
 11  TIPOPROCESSO                 23546 non-null  o

####Verificando distribuição dos dados propriamente dita

In [ ]:
total   = dataset.shape[0]
label_s = dataset[dataset['DECISAO_FAVORAVEL'] == 'S'].shape[0]
label_n = dataset[dataset['DECISAO_FAVORAVEL'] == 'N'].shape[0]

print(f'TOTAL  : {total}')
print(f'LABEL S: {label_s} - % {round(label_s/total*100, 2)}')
print(f'LABEL N: {label_n} - % {round(label_n/total*100, 2)}')

TOTAL  : 404578
LABEL S: 76011 - % 18.79
LABEL N: 327930 - % 81.05


In [ ]:
descricao_periodo = 'geral' if (ANO_CARGA == None) else ANO_CARGA
descricao_periodo

'geral'

In [ ]:
CLASSE

'HC'

In [ ]:
import pandas as pd

# Nova regra para descricao_periodo: ANO_CARGA ou intervalo ANO_INI_ANO_FIM
if ANO_CARGA is not None:
    descricao_periodo = str(ANO_CARGA)
elif ANO_INI is not None and ANO_FIM is not None:
    descricao_periodo = f"{ANO_INI}_{ANO_FIM}"
else:
    descricao_periodo = "geral"

tipos_str = '_'.join(LISTA_TIPO_ARQUIVO)

# Conditionally add NROWS to the filename if it's set and not None or 0
nrows_suffix = f"_NROWS_{NROWS}" if NROWS is not None and NROWS > 0 else ""

output_csv_path = f"{base_path}dataset_{tipos_str}_{CLASSE}_{descricao_periodo}{nrows_suffix}.csv"
print(f"Caminho de saída: {output_csv_path}")

# Salvar o DataFrame
dataset.to_csv(output_csv_path, index=False, sep=';')

print(f"Dataset pré-processado salvo em: {output_csv_path}")
display(dataset.head())

Caminho de saída: /content/drive/MyDrive/_Mestrado/Datasets/dataset_decisoes_acordaos_HC_2021_2025.csv
Dataset pré-processado salvo em: /content/drive/MyDrive/_Mestrado/Datasets/dataset_decisoes_acordaos_HC_2021_2025.csv


,SEQDOCUMENTO,DATAPUBLICACAO,TIPODOCUMENTO,NUMEROREGISTRO,DATARECEBIMENTO,DATADISTRIBUICAO,MINISTRO,RECURSO,TEOR,ASSUNTOS,NUM_DIAS_CONCLUSAO_PROCESSO,TIPOPROCESSO,NUMEROPROCESSO,DECISAO_FAVORAVEL,RELATORIO
0,353105536,2025-12-31,DECISÃO,202505102484,2025-12-20,2025-12-23,HERMAN BENJAMIN,NaN,Negando,4355,11,HC,1063382,N,Cuida-se de Habeas Corpus com pedido de limina...
1,353128400,2025-12-31,DECISÃO,202505106071,2025-12-22,2025-12-26,HERMAN BENJAMIN,NaN,Negando,5897,9,HC,1063851,N,Trata-se de Habeas Corpus com pedido de limina...
2,353128835,2025-12-31,DECISÃO,202505109521,2025-12-23,2025-12-26,HERMAN BENJAMIN,NaN,Negando,3608,8,HC,1064262,N,Cuida-se de Habeas Corpus com pedido de limina...
3,353107646,2025-12-31,DECISÃO,202505102649,2025-12-21,2025-12-24,HERMAN BENJAMIN,NaN,Negando,4355,10,HC,1063449,N,Cuida-se de Habeas Corpus com pedido de limina...
4,352928478,2025-12-31,DECISÃO,202505059266,2025-12-18,2025-12-19,RIBEIRO DANTAS,NaN,Negando,10635,13,HC,1062801,N,"Trata-se de habeas corpus, com pedido de limin..."


In [ ]:
import pandas as pd

# Define the path to the preprocessed CSV file
input_csv_path = output_csv_path

# Load the CSV file into a DataFrame
try:
    df_loaded = pd.read_csv(input_csv_path, sep=';')
    print(f"Successfully loaded {input_csv_path} into dataframe 'df_loaded'")
    display(df_loaded.head())
except FileNotFoundError:
    print(f"Error: File not found at {input_csv_path}")
except Exception as e:
    print(f"An error occurred while loading {input_csv_path}: {e}")

Successfully loaded /content/drive/MyDrive/_Mestrado/Datasets/dataset_decisoes_acordaos_HC_2021_2025.csv into dataframe 'df_loaded'


,SEQDOCUMENTO,DATAPUBLICACAO,TIPODOCUMENTO,NUMEROREGISTRO,DATARECEBIMENTO,DATADISTRIBUICAO,MINISTRO,RECURSO,TEOR,ASSUNTOS,NUM_DIAS_CONCLUSAO_PROCESSO,TIPOPROCESSO,NUMEROPROCESSO,DECISAO_FAVORAVEL,RELATORIO
0,353105536,2025-12-31,DECISÃO,202505102484,2025-12-20,2025-12-23,HERMAN BENJAMIN,NaN,Negando,4355,11,HC,1063382,N,Cuida-se de Habeas Corpus com pedido de limina...
1,353128400,2025-12-31,DECISÃO,202505106071,2025-12-22,2025-12-26,HERMAN BENJAMIN,NaN,Negando,5897,9,HC,1063851,N,Trata-se de Habeas Corpus com pedido de limina...
2,353128835,2025-12-31,DECISÃO,202505109521,2025-12-23,2025-12-26,HERMAN BENJAMIN,NaN,Negando,3608,8,HC,1064262,N,Cuida-se de Habeas Corpus com pedido de limina...
3,353107646,2025-12-31,DECISÃO,202505102649,2025-12-21,2025-12-24,HERMAN BENJAMIN,NaN,Negando,4355,10,HC,1063449,N,Cuida-se de Habeas Corpus com pedido de limina...
4,352928478,2025-12-31,DECISÃO,202505059266,2025-12-18,2025-12-19,RIBEIRO DANTAS,NaN,Negando,10635,13,HC,1062801,N,"Trata-se de habeas corpus, com pedido de limin..."


In [ ]:
df_loaded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404578 entries, 0 to 404577
Data columns (total 15 columns):
 #   Column                       Non-Null Count   Dtype 
---  ------                       --------------   ----- 
 0   SEQDOCUMENTO                 404578 non-null  int64 
 1   DATAPUBLICACAO               404578 non-null  object
 2   TIPODOCUMENTO                404578 non-null  object
 3   NUMEROREGISTRO               404578 non-null  int64 
 4   DATARECEBIMENTO              404578 non-null  object
 5   DATADISTRIBUICAO             404571 non-null  object
 6   MINISTRO                     404578 non-null  object
 7   RECURSO                      68147 non-null   object
 8   TEOR                         403941 non-null  object
 9   ASSUNTOS                     404578 non-null  object
 10  NUM_DIAS_CONCLUSAO_PROCESSO  404578 non-null  int64 
 11  TIPOPROCESSO                 404578 non-null  object
 12  NUMEROPROCESSO               404578 non-null  int64 
 13  DECISAO_FAVORA

In [ ]:
df_loaded.shape

(404578, 15)

In [ ]:
!python --version

Python 3.13.15


In [ ]:
import pandas as pd
import numpy as np
import torch

print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")
print(f"PyTorch version: {torch.__version__}")

Pandas version: 2.2.3
NumPy version: 2.1.3
PyTorch version: 2.11.0+cpu
